In [2]:
import os
import time
from typing import Any, Callable
from dotenv import load_dotenv

# Nạp OPENAI_API_KEY từ file .env (copy .env.example thành .env và dán key vào)
load_dotenv()

# ---------------------------------------------------------------------------
# Bảng giá ước tính (USD / 1K token) — cập nhật nếu giá thay đổi
# ---------------------------------------------------------------------------
PRICING_PER_1K_TOKENS = {
    "gpt-4o": {"input": 0.0025, "output": 0.010},
    "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
    "gemini-3.5-flash": {"input": 0.0003, "output": 0.0025},
    "gemini-3.5-flash-lite": {"input": 0.0001, "output": 0.0004},
}

# Luồng chính: OpenAI (mặc định, không cần đặt gì trong .env).
# Không có key OpenAI? Dùng luồng thay thế Google Gemini (Phụ lục B
# trong LAB_GUIDE.md) — tên model đổi qua .env. NVIDIA NIM: Phụ lục C.
OPENAI_MODEL = os.getenv("LAB_MODEL", "gpt-4o")
OPENAI_MINI_MODEL = os.getenv("LAB_MINI_MODEL", "gpt-4o-mini")



In [3]:


def call_openai(
    prompt: str,
    model: str = OPENAI_MODEL,
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_tokens: int = 256,
) -> tuple[str, float]:
    """
    Gọi OpenAI Chat Completions API, trả về nội dung phản hồi + độ trễ.

    Args:
        prompt:      Tin nhắn của người dùng.
        model:       Model OpenAI sử dụng (mặc định: gpt-4o).
        temperature: Độ ngẫu nhiên khi lấy mẫu (0.0 – 2.0).
        top_p:       Ngưỡng nucleus sampling.
        max_tokens:  Số token tối đa được sinh ra.

    Returns:
        Tuple (response_text: str, latency_seconds: float).

    Gợi ý:
        from openai import OpenAI            # import BÊN TRONG hàm
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        # đo thời gian bằng time.time() trước và sau lời gọi API
    """
    # TODO: import OpenAI, tạo client, gọi chat.completions.create,
    #       đo start/end time, trả về (response_text, latency)

    from openai import OpenAI

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=max_tokens,
    )

    latency = time.time() - start 

    return response.choices[0].message.content, latency


# ---------------------------------------------------------------------------
# Task 1.2 — Gọi GPT-4o-mini
# ---------------------------------------------------------------------------
def call_openai_mini(
    prompt: str,
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_tokens: int = 256,
) -> tuple[str, float]:
    """
    Gọi API với model gpt-4o-mini — nhanh hơn và rẻ hơn.

    Returns:
        Tuple (response_text: str, latency_seconds: float).

    Gợi ý:
        Tái sử dụng call_openai() với model=OPENAI_MINI_MODEL — 1 dòng code.
    """
    # TODO: gọi call_openai với model=OPENAI_MINI_MODEL
    return call_openai(
    prompt=prompt,
    model=OPENAI_MINI_MODEL,
    temperature=temperature,
    top_p=top_p,
    max_tokens=max_tokens,
    )


# ---------------------------------------------------------------------------
# Task 1.3 — So sánh GPT-4o vs GPT-4o-mini
# ---------------------------------------------------------------------------
def compare_models(prompt: str) -> dict:
    """
    Gọi cả hai model với cùng một prompt và trả về dict so sánh.

    Returns:
        Dict với các key:
            - "gpt4o_answer":      str
            - "mini_answer":       str
            - "gpt4o_time":       float
            - "mini_time":        float
            - "gpt4o_cost": float  (USD ước tính cho phản hồi)

    Gợi ý:
        pricing = PRICING_PER_1K_TOKENS.get(
            OPENAI_MODEL, PRICING_PER_1K_TOKENS["gpt-4o"]
        )
        cost = (len(response.split()) / 0.75) / 1000 * pricing["output"]
        (0.75 từ ≈ 1 token — ước lượng thô; Part 2 sẽ tính chính xác hơn.
         Dùng .get để lấy đúng giá model đang chạy — gpt-4o, gemini...;
         model không có trong bảng thì lấy giá gpt-4o làm tham chiếu)
    """
    # TODO: gọi call_openai và call_openai_mini, ghép dict kết quả
    gpt4o_text, gpt4o_time = call_openai(prompt)
    mini_text, mini_time = call_openai_mini(prompt)

    pricing = PRICING_PER_1K_TOKENS.get(
        OPENAI_MODEL,
        PRICING_PER_1K_TOKENS["gpt-4o"]
    )

    gpt4o_cost = (
        (len(gpt4o_text.split()) / 0.75) / 1000
    ) * pricing["output"]

    return {
        "gpt4o_answer": gpt4o_text,
        "mini_answer": mini_text,
        "gpt4o_time": gpt4o_time,
        "mini_time": mini_time,
        "gpt4o_cost": gpt4o_cost,
    }

In [ ]:
prompt = "Hãy kể cho tôi một sự thật thú vị về Hà Nội."
temperatures = [0.0, 0.7, 1.2, 1.8]

for temp in temperatures:
    response, latency = call_openai(prompt=prompt, temperature=temp, max_tokens=2000)
    print(f"Temperature = {temp}")
    print(response)
    print(f"Latency: {latency:.2f}s")
    print("-" * 50)

In [ ]:
def chat_with_system_prompt(
    system_prompt: str,
    user_prompt: str,
    model: str = OPENAI_MODEL,
    temperature: float = 0.7,
    max_tokens: int = 256,
) -> tuple[str, float]:
    """
    Gọi API với MESSAGES gồm 2 phần: system prompt (định hình vai trò/persona
    của model) và user prompt (câu hỏi thật).

    Args:
        system_prompt: Chỉ dẫn vai trò, ví dụ "Bạn là giáo viên tiểu học,
                       giải thích mọi thứ thật đơn giản."
        user_prompt:   Tin nhắn của người dùng.

    Returns:
        Tuple (response_text: str, latency_seconds: float).

    Gợi ý:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    """
    # TODO: giống call_openai nhưng messages có thêm phần tử role="system"
    from openai import OpenAI
    
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    
    start = time.time()
    
    response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_completion_tokens=max_tokens,
        )
    
    latency = time.time() - start 
    
    return response.choices[0].message.content, latency


In [ ]:
system_prompt1 = "Bạn là một nhà thơ, trả lời mọi thứ bằng hình ảnh ví von, tránh thuật ngữ."
system_prompt2 = "Bạn là kỹ sư phần mềm senior, trả lời chính xác, có ví dụ code khi phù hợp."
prompt = "Giải thích máy học (machine learning) là gì?"
response1 = chat_with_system_prompt(system_prompt1, prompt, max_tokens=1000)


NameError: name 'chat_with_system_prompt' is not defined

In [ ]:
response2 = chat_with_system_prompt(system_prompt2, prompt, max_tokens=1000)

In [5]:
def count_tokens(text: str, model: str = OPENAI_MODEL) -> int:
    """
    Đếm số token của một đoạn text bằng thư viện tiktoken.

    Args:
        text:  Đoạn text cần đếm.
        model: Model dùng để chọn bộ mã hóa (encoding).

    Returns:
        Số token (int).

    Gợi ý:
        import tiktoken
        enc = tiktoken.encoding_for_model(model)
        return len(enc.encode(text))

        tiktoken cần tải bộ mã hóa từ mạng ở lần chạy đầu. Hãy bọc trong
        try/except — nếu lỗi (offline, model lạ), dùng ước lượng dự phòng:
        max(1, len(text) // 4)   (trung bình 1 token ≈ 4 ký tự)
    """
    # TODO: dùng tiktoken để đếm token, có fallback khi lỗi
    try:
        import socket
        socket.setdefaulttimeout(3)  # tối đa 3 giây chờ mạng
        import tiktoken
        enc = tiktoken.encoding_for_model(model)
        return len(enc.encode(text))
    except Exception:
        return max(1, len(text) // 4)

In [6]:
pharagraph = "Trong kịch bản có 20.000 người dùng hoạt động mỗi ngày, mỗi người gọi API hai lần và mỗi lần tạo khoảng 500 output token, hệ thống sẽ tiêu thụ khoảng 20 triệu output token mỗi ngày. Theo bảng giá trong đề, chi phí sử dụng model nhỏ chỉ khoảng **2 USD/ngày**, trong khi model lớn như GPT-4o vào khoảng **50 USD/ngày**, tức cao hơn khoảng 25 lần. Tuy nhiên, việc lựa chọn model không nên chỉ dựa trên chi phí mà còn cần cân nhắc hiệu quả mang lại. Model lớn phù hợp với các tác vụ đòi hỏi khả năng suy luận và độ chính xác cao như soạn thảo hợp đồng pháp lý, phân tích tài liệu hoặc hỗ trợ lập trình, nơi chất lượng câu trả lời có ảnh hưởng trực tiếp đến kết quả công việc. Ngược lại, model nhỏ là lựa chọn hợp lý cho chatbot chăm sóc khách hàng, trả lời câu hỏi thường gặp hoặc tóm tắt văn bản, vì vẫn đáp ứng tốt nhu cầu với chi phí thấp và khả năng mở rộng cao."
count_tokens(pharagraph)


215